# Mixture of Experts: The Router

In a Mixture of Experts (MoE) model, we don't use all parameters for every token. Instead, a **Router** decides which "Experts" (small neural networks) should process each token.

## Top-K Routing

We use a Top-K router, which selects the best $k$ experts for each token based on a learned gating mechanism.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.top_k = top_k

    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        
        # 1. Compute scores for each expert
        logits = self.gate(x)
        
        # 2. Select top-k experts
        top_k_logits, indices = torch.topk(logits, self.top_k, dim=-1)
        
        # 3. Normalize scores to get weights (probabilities)
        weights = F.softmax(top_k_logits, dim=-1)
        
        return weights, indices

# Test it
d_model = 16
num_experts = 4
router = TopKRouter(d_model, num_experts)
x = torch.randn(1, 5, d_model) # Batch 1, Seq 5

weights, indices = router(x)
print("Selected Experts Indices:", indices)
print("Expert Weights:", weights)